In [25]:
# import pandas as pd
# import numpy as np
# import openpyxl 
# import os

# # --- تنظیمات آدرس‌ها ---
# file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output2.xlsx'

# # سنسورهای هدف
# target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# def run_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
#         return

#     try:
#         df = pd.read_excel(file_path, parse_dates=['date'])
#         df.set_index('date', inplace=True)
#         print("✅ مرحله ۱: فایل بارگذاری شد.")
#     except Exception as e:
#         print(f"❌ خطا در خواندن اکسل: {e}")
#         return

#     # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر (Fault)
#     try:
#         last_date = df.index.max()
#         split_date = last_date - pd.Timedelta(days=30)
#         baseline_start = split_date - pd.Timedelta(days=30)

#         df_baseline = df.loc[baseline_start:split_date].copy()
#         df_fault = df.loc[split_date:last_date].copy()
#     except Exception as e:
#         print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
#         return

#     # ۲. محاسبه شیب و TTT برای یک ماه اخیر
#     print("⏳ در حال محاسبه شیب تغییرات و زمان تخمینی برای سنسورهای هدف...")

#     target_analysis_results = []

#     for col in target_sensors:
#         if col not in df_fault.columns: continue

#         try:
#             # الف) حد آستانه از بازه سلامت
#             mean_base = df_baseline[col].mean()
#             std_base = df_baseline[col].std()
#             upper_limit = mean_base + 3 * std_base 

#             # ب) محاسبه شیب (تغییرات در هر ساعت)
#             time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
#             val_diff = df_fault[col].diff()

#             instant_slopes = val_diff / time_diff_series
#             avg_slope = instant_slopes.mean()

#             current_val = df_fault[col].iloc[-1] 

#             # ج) محاسبه TTT (ساعت مانده تا حد بحرانی)
#             ttt_hours = np.nan
#             if avg_slope > 0 and current_val < upper_limit:
#                 ttt_hours = (upper_limit - current_val) / avg_slope

#             target_analysis_results.append({
#                 'Sensor': col,
#                 'Current_Value': round(current_val, 4),
#                 'Baseline_Limit(3Sigma)': round(upper_limit, 4),
#                 'Average_Slope_per_Hour': round(avg_slope, 6),
#                 'Hours_to_Threshold': round(ttt_hours, 2) if (not np.isnan(ttt_hours) and ttt_hours != np.inf) else "No Risk"
#             })
#         except: continue

#     # ۳. رتبه‌بندی RCA
#     numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
#     rca_list = []
#     for c in numeric_cols:
#         try:
#             dev = abs(df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() + 1e-6)
#             rca_list.append({'Sensor': c, 'Deviation_Score': round(dev, 4)})
#         except: continue
#     rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)

#     # ۴. ذخیره در اکسل
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)

#         with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#             pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
#             rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)

#         print(f"🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
#     except Exception as e:
#         print(f"❌ خطا در ذخیره فایل: {e}")

# # فراخوانی مستقیم تابع برای جلوگیری از خطای NameError
# run_analysis()

In [26]:
# import pandas as pd
# import numpy as np
# import openpyxl 
# import os

# # --- تنظیمات آدرس‌ها ---
# file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output2.xlsx'

# # سنسورهای هدف
# target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# # پارامتر لاندا (ضریب وزنی) - بر اساس حساسیت مورد نظر تنظیم می‌شود
# LAMBDA = 0.2  # معمولاً بین 0.05 تا 0.3

# def calculate_ewma(data, lambda_val):
#     """محاسبه مقادیر EWMA به صورت بازگشتی"""
#     ewma_values = np.zeros(len(data))
#     if len(data) == 0:
#         return ewma_values
    
#     # مقدار اولیه برابر با اولین داده است
#     ewma_values[0] = data[0]
    
#     for t in range(1, len(data)):
#         ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
    
#     return ewma_values

# def calculate_control_limits(mean, std, lambda_val):
#     """محاسبه UCL و LCL بر اساس فرمول EWMA در حالت پایدار"""
#     factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
#     ucl = mean + factor
#     lcl = mean - factor
#     return ucl, lcl

# def run_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
#         return

#     try:
#         df = pd.read_excel(file_path, parse_dates=['date'])
#         df.set_index('date', inplace=True)
#         print("✅ مرحله ۱: فایل بارگذاری شد.")
#         print(f"📅 بازه زمانی داده‌ها: {df.index.min()} تا {df.index.max()}")
#     except Exception as e:
#         print(f"❌ خطا در خواندن اکسل: {e}")
#         return

#     # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر (Fault)
#     try:
#         last_date = df.index.max()
#         split_date = last_date - pd.Timedelta(days=30)
#         baseline_start = split_date - pd.Timedelta(days=30)

#         df_baseline = df.loc[baseline_start:split_date].copy()
#         df_fault = df.loc[split_date:last_date].copy()
        
#         print(f"📊 بازه سلامت (بیس‌لاین): {baseline_start.date()} تا {split_date.date()}")
#         print(f"⚠️ بازه خطا (یک ماه اخیر): {split_date.date()} تا {last_date.date()}")
        
#     except Exception as e:
#         print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
#         return

#     # ۲. محاسبه شیب، EWMA، UCL/LCL و TTT برای سنسورهای هدف
#     print("⏳ در حال محاسبه شاخص‌های EWMA، خطوط کنترل و زمان تخمینی برای سنسورهای هدف...")
#     print(f"🔧 مقدار لاندا (λ) = {LAMBDA}")

#     target_analysis_results = []

#     for col in target_sensors:
#         if col not in df_fault.columns:
#             print(f"⚠️ سنسور {col} در داده‌ها وجود ندارد.")
#             continue
        
#         if col not in df_baseline.columns:
#             print(f"⚠️ سنسور {col} در داده‌های بیس‌لاین وجود ندارد.")
#             continue

#         try:
#             # ---- الف) محاسبه پارامترهای آماری از بازه سلامت ----
#             mean_base = df_baseline[col].mean()
#             std_base = df_baseline[col].std()
            
#             # محاسبه UCL و LCL بر اساس فرمول EWMA
#             ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
            
#             # ---- ب) محاسبه مقادیر EWMA برای بازه خطا ----
#             fault_data = df_fault[col].values
#             ewma_values = calculate_ewma(fault_data, LAMBDA)
            
#             # آخرین مقدار EWMA
#             current_ewma = ewma_values[-1]
            
#             # ---- ج) محاسبه شیب تغییرات (بر اساس داده‌های خام، نه EWMA) ----
#             time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
#             val_diff = df_fault[col].diff()
#             instant_slopes = val_diff / time_diff_series
#             avg_slope = instant_slopes.mean()
            
#             current_raw_value = df_fault[col].iloc[-1]
            
#             # ---- د) محاسبه TTT (ساعت مانده تا حد بحرانی UCL) ----
#             ttt_hours = np.nan
#             if avg_slope > 0 and current_raw_value < ucl:
#                 ttt_hours = (ucl - current_raw_value) / avg_slope
#                 ttt_hours = round(ttt_hours, 2)
#             elif avg_slope <= 0:
#                 ttt_hours = 0  # در شیب منفی یا صفر، زمان رسیدن به حد بالایی بی‌نهایت است، صفر قرار می‌دهیم
#             else:
#                 ttt_hours = "No Risk"
            
#             # بررسی اینکه آیا مقدار EWMA خارج از محدوده کنترل است
#             out_of_control = "No"
#             if current_ewma > ucl or current_ewma < lcl:
#                 out_of_control = "Yes ⚠️"
            
#             target_analysis_results.append({
#                 'Sensor': col,
#                 'Current_Raw_Value': round(current_raw_value, 4),
#                 'Current_EWMA': round(current_ewma, 4),
#                 'UCL': round(ucl, 4),
#                 'LCL': round(lcl, 4),
#                 'Baseline_Mean': round(mean_base, 4),
#                 'Baseline_Std': round(std_base, 4),
#                 'Average_Slope_per_Hour': round(avg_slope, 6),
#                 'Hours_to_UCL': ttt_hours,
#                 'Out_Of_Control': out_of_control
#             })
            
#             # چاپ وضعیت برای مانیتورینگ
#             status = "✅ نرمال" if out_of_control == "No" else "⚠️ خارج از کنترل"
#             print(f"  📊 {col}: EWMA={round(current_ewma,4)}, UCL={round(ucl,4)}, LCL={round(lcl,4)} → {status}")
            
#         except Exception as e:
#             print(f"❌ خطا در پردازش سنسور {col}: {e}")
#             continue

#     # ۳. رتبه‌بندی RCA (بر اساس انحراف معیار)
#     print("⏳ در حال محاسبه رتبه‌بندی RCA برای همه سنسورها...")
    
#     numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
#     rca_list = []
#     for c in numeric_cols:
#         if c not in df_fault.columns:
#             continue
#         try:
#             # محاسبه انحراف استاندارد شده (نمره Z)
#             mean_base = df_baseline[c].mean()
#             std_base = df_baseline[c].std()
#             mean_fault = df_fault[c].mean()
            
#             if std_base > 0:
#                 deviation_score = abs(mean_fault - mean_base) / std_base
#             else:
#                 deviation_score = 0
                
#             rca_list.append({
#                 'Sensor': c, 
#                 'Baseline_Mean': round(mean_base, 4),
#                 'Fault_Mean': round(mean_fault, 4),
#                 'Deviation_Score': round(deviation_score, 4)
#             })
#         except Exception as e:
#             print(f"⚠️ خطا در محاسبه RCA برای {c}: {e}")
#             continue
    
#     rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
#     # اضافه کردن توضیحات روش محاسبه
#     methodology_note = pd.DataFrame({
#         'Parameter': ['Lambda (λ)', 'UCL Formula', 'LCL Formula', 'EWMA Formula', 'Control Limit Factor'],
#         'Value': [f'{LAMBDA}', f'μ + 3σ√(λ/(2-λ))', f'μ - 3σ√(λ/(2-λ))', 'E_t = λ·X_t + (1-λ)·E_{t-1}', '3σ (Based on ISO 7870-4)']
#     })

#     # ۴. ذخیره در اکسل
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)

#         with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#             pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
#             rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
#             methodology_note.to_excel(writer, sheet_name='Methodology', index=False)

#         print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
#         print(f"\n📋 خلاصه خروجی:")
#         print(f"   - Speed_Analysis: تحلیل {len(target_analysis_results)} سنسور هدف با شاخص‌های EWMA")
#         print(f"   - RCA_Ranking: رتبه‌بندی {len(rca_summary)} سنسور بر اساس انحراف")
#         print(f"   - Methodology: پارامترها و فرمول‌های استفاده شده")
        
#     except Exception as e:
#         print(f"❌ خطا در ذخیره فایل: {e}")

# # فراخوانی تابع
# if __name__ == "__main__":
#     run_analysis()

In [27]:
# import pandas as pd
# import numpy as np
# import openpyxl 
# import os

# # --- تنظیمات آدرس‌ها ---
# file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output2.xlsx'

# # سنسورهای هدف
# target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# # پارامتر لاندا (ضریب وزنی) - بر اساس حساسیت مورد نظر تنظیم می‌شود
# LAMBDA = 0.2  # معمولاً بین 0.05 تا 0.3

# def calculate_ewma(data, lambda_val):
#     """محاسبه مقادیر EWMA به صورت بازگشتی"""
#     ewma_values = np.zeros(len(data))
#     if len(data) == 0:
#         return ewma_values
    
#     # مقدار اولیه برابر با اولین داده است
#     ewma_values[0] = data[0]
    
#     for t in range(1, len(data)):
#         ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
    
#     return ewma_values

# def calculate_control_limits(mean, std, lambda_val):
#     """محاسبه UCL و LCL بر اساس فرمول EWMA در حالت پایدار"""
#     factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
#     ucl = mean + factor
#     lcl = mean - factor
#     return ucl, lcl

# def run_analysis():
#     if not os.path.exists(file_path):
#         print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
#         return

#     try:
#         df = pd.read_excel(file_path, parse_dates=['date'])
#         df.set_index('date', inplace=True)
#         print("✅ مرحله ۱: فایل بارگذاری شد.")
#         print(f"📅 بازه زمانی داده‌ها: {df.index.min()} تا {df.index.max()}")
#     except Exception as e:
#         print(f"❌ خطا در خواندن اکسل: {e}")
#         return

#     # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر (Fault)
#     try:
#         last_date = df.index.max()
#         split_date = last_date - pd.Timedelta(days=30)
#         baseline_start = split_date - pd.Timedelta(days=30)

#         df_baseline = df.loc[baseline_start:split_date].copy()
#         df_fault = df.loc[split_date:last_date].copy()
        
#         print(f"📊 بازه سلامت (بیس‌لاین): {baseline_start.date()} تا {split_date.date()}")
#         print(f"⚠️ بازه خطا (یک ماه اخیر): {split_date.date()} تا {last_date.date()}")
        
#     except Exception as e:
#         print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
#         return

#     # ۲. محاسبه شیب، EWMA، UCL/LCL و TTT برای سنسورهای هدف
#     print("⏳ در حال محاسبه شاخص‌های EWMA، خطوط کنترل و زمان تخمینی برای سنسورهای هدف...")
#     print(f"🔧 مقدار لاندا (λ) = {LAMBDA}")

#     target_analysis_results = []

#     for col in target_sensors:
#         if col not in df_fault.columns:
#             print(f"⚠️ سنسور {col} در داده‌ها وجود ندارد.")
#             continue
        
#         if col not in df_baseline.columns:
#             print(f"⚠️ سنسور {col} در داده‌های بیس‌لاین وجود ندارد.")
#             continue

#         try:
#             # ---- الف) محاسبه پارامترهای آماری از بازه سلامت ----
#             mean_base = df_baseline[col].mean()
#             std_base = df_baseline[col].std()
            
#             # محاسبه UCL و LCL بر اساس فرمول EWMA
#             ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
            
#             # ---- ب) محاسبه مقادیر EWMA برای بازه خطا ----
#             fault_data = df_fault[col].values
#             ewma_values = calculate_ewma(fault_data, LAMBDA)
            
#             # آخرین مقدار EWMA
#             current_ewma = ewma_values[-1]
            
#             # ---- ج) محاسبه شیب تغییرات (بر اساس داده‌های خام، نه EWMA) ----
#             time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
#             val_diff = df_fault[col].diff()
#             instant_slopes = val_diff / time_diff_series
#             avg_slope = instant_slopes.mean()
            
#             current_raw_value = df_fault[col].iloc[-1]
            
#             # ---- د) محاسبه Hours_to_UCL (فقط عدد، ترجیحاً عدد صحیح یا با دو رقم اعشار) ----
#             # منطق ساده و صریح:
#             # اگر شیب متوسط <= 0 باشد -> 0 ساعت (هیچ روند افزایشی نداریم)
#             # اگر شیب مثبت باشد:
#             #   - اگر مقدار فعلی >= UCL باشد -> 0 ساعت (از حد عبور کرده)
#             #   - اگر مقدار فعلی < UCL باشد -> (UCL - مقدار فعلی) / شیب
#             # در نهایت مقدار را به عدد (با دو رقم اعشار) تبدیل می‌کنیم
            
#             if avg_slope <= 0:
#                 hours_to_ucl = 0.0
#             else:  # شیب مثبت
#                 if current_raw_value >= ucl:
#                     hours_to_ucl = 0.0
#                 else:
#                     hours_to_ucl = (ucl - current_raw_value) / avg_slope
                    
#             # اطمینان از اینکه خروجی عدد است (نه NaN، نه متن)
#             # اگر به هر دلیلی NaN شد، صفر می‌گذاریم
#             if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
#                 hours_to_ucl = 0.0
#             else:
#                 hours_to_ucl = round(hours_to_ucl, 2)  # دو رقم اعشار
            
#             # بررسی اینکه آیا مقدار EWMA خارج از محدوده کنترل است
#             out_of_control = "No"
#             if current_ewma > ucl or current_ewma < lcl:
#                 out_of_control = "Yes ⚠️"
            
#             target_analysis_results.append({
#                 'Sensor': col,
#                 'Current_Raw_Value': round(current_raw_value, 4),
#                 'Current_EWMA': round(current_ewma, 4),
#                 'UCL': round(ucl, 4),
#                 'LCL': round(lcl, 4),
#                 'Baseline_Mean': round(mean_base, 4),
#                 'Baseline_Std': round(std_base, 4),
#                 'Average_Slope_per_Hour': round(avg_slope, 6),
#                 'Hours_to_UCL': hours_to_ucl,  # ✅ همیشه عدد (0 یا مثبت)
#                 'Out_Of_Control': out_of_control
#             })
            
#             # چاپ وضعیت برای مانیتورینگ
#             status = "✅ نرمال" if out_of_control == "No" else "⚠️ خارج از کنترل"
#             print(f"  📊 {col}: مقدار={round(current_raw_value,4)}, UCL={round(ucl,4)}, شیب={round(avg_slope,6)}/hr → Hours_to_UCL={hours_to_ucl} ساعت → {status}")
            
#         except Exception as e:
#             print(f"❌ خطا در پردازش سنسور {col}: {e}")
#             continue

#     # ۳. رتبه‌بندی RCA (بر اساس انحراف معیار)
#     print("⏳ در حال محاسبه رتبه‌بندی RCA برای همه سنسورها...")
    
#     numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
#     rca_list = []
#     for c in numeric_cols:
#         if c not in df_fault.columns:
#             continue
#         try:
#             # محاسبه انحراف استاندارد شده (نمره Z)
#             mean_base = df_baseline[c].mean()
#             std_base = df_baseline[c].std()
#             mean_fault = df_fault[c].mean()
            
#             if std_base > 0:
#                 deviation_score = abs(mean_fault - mean_base) / std_base
#             else:
#                 deviation_score = 0
                
#             rca_list.append({
#                 'Sensor': c, 
#                 'Baseline_Mean': round(mean_base, 4),
#                 'Fault_Mean': round(mean_fault, 4),
#                 'Deviation_Score': round(deviation_score, 4)
#             })
#         except Exception as e:
#             print(f"⚠️ خطا در محاسبه RCA برای {c}: {e}")
#             continue
    
#     rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
#     # اضافه کردن توضیحات روش محاسبه
#     methodology_note = pd.DataFrame({
#         'Parameter': ['Lambda (λ)', 'UCL Formula', 'LCL Formula', 'EWMA Formula', 'Control Limit Factor', 'Hours_to_UCL Rule'],
#         'Value': [
#             f'{LAMBDA}', 
#             f'μ + 3σ√(λ/(2-λ))', 
#             f'μ - 3σ√(λ/(2-λ))', 
#             'E_t = λ·X_t + (1-λ)·E_{t-1}', 
#             '3σ (Based on ISO 7870-4)',
#             'If slope<=0 → 0 | If slope>0 & value<UCL → (UCL-value)/slope | else → 0'
#         ]
#     })

#     # ۴. ذخیره در اکسل
#     try:
#         os.makedirs(os.path.dirname(output_filename), exist_ok=True)

#         with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#             pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
#             rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
#             methodology_note.to_excel(writer, sheet_name='Methodology', index=False)

#         print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
#         print(f"\n📋 خلاصه خروجی:")
#         print(f"   - Speed_Analysis: تحلیل {len(target_analysis_results)} سنسور هدف")
#         print(f"   - RCA_Ranking: رتبه‌بندی {len(rca_summary)} سنسور بر اساس انحراف")
#         print(f"   - Methodology: پارامترها و فرمول‌های استفاده شده")
        
#         # نمایش نمونه خروجی Hours_to_UCL
#         if len(target_analysis_results) > 0:
#             print(f"\n📊 نمونه مقادیر Hours_to_UCL:")
#             for res in target_analysis_results[:3]:
#                 print(f"   {res['Sensor']}: شیب={res['Average_Slope_per_Hour']} → Hours_to_UCL={res['Hours_to_UCL']} ساعت")
        
#     except Exception as e:
#         print(f"❌ خطا در ذخیره فایل: {e}")

# # فراخوانی تابع
# if __name__ == "__main__":
#     run_analysis()

In [28]:
import pandas as pd
import numpy as np
import openpyxl 
import os

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output2.xlsx'

# سنسورهای هدف
target_sensors = ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# پارامتر لاندا (ضریب وزنی)
LAMBDA = 0.2

def calculate_ewma(data, lambda_val):
    """محاسبه مقادیر EWMA به صورت بازگشتی"""
    ewma_values = np.zeros(len(data))
    if len(data) == 0:
        return ewma_values
    
    ewma_values[0] = data[0]
    for t in range(1, len(data)):
        ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
    
    return ewma_values

def calculate_control_limits(mean, std, lambda_val):
    """محاسبه UCL و LCL بر اساس فرمول EWMA در حالت پایدار"""
    factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
    ucl = mean + factor
    lcl = mean - factor
    return ucl, lcl

def run_analysis():
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
        return

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print("✅ مرحله ۱: فایل بارگذاری شد.")
        print(f"📅 بازه زمانی داده‌ها: {df.index.min()} تا {df.index.max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return

    # ۱. جداسازی بازه سلامت و بازه یک ماه اخیر
    try:
        last_date = df.index.max()
        split_date = last_date - pd.Timedelta(days=30)
        baseline_start = split_date - pd.Timedelta(days=30)

        df_baseline = df.loc[baseline_start:split_date].copy()
        df_fault = df.loc[split_date:last_date].copy()
        
        print(f"📊 بازه سلامت: {baseline_start.date()} تا {split_date.date()}")
        print(f"⚠️ بازه خطا: {split_date.date()} تا {last_date.date()}")
        
    except Exception as e:
        print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
        return

    print("⏳ در حال محاسبه شاخص‌ها...")
    print(f"🔧 لاندا (λ) = {LAMBDA}")
    print("="*80)

    target_analysis_results = []

    for col in target_sensors:
        if col not in df_fault.columns:
            print(f"⚠️ سنسور {col} در داده‌ها وجود ندارد.")
            continue
        
        if col not in df_baseline.columns:
            print(f"⚠️ سنسور {col} در داده‌های بیس‌لاین وجود ندارد.")
            continue

        try:
            # محاسبه پارامترهای آماری از بازه سلامت
            mean_base = df_baseline[col].mean()
            std_base = df_baseline[col].std()
            
            # محاسبه UCL و LCL
            ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
            
            # محاسبه مقادیر EWMA برای بازه خطا
            fault_data = df_fault[col].values
            ewma_values = calculate_ewma(fault_data, LAMBDA)
            current_ewma = ewma_values[-1]
            
            # محاسبه شیب تغییرات
            time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
            val_diff = df_fault[col].diff()
            instant_slopes = val_diff / time_diff_series
            avg_slope = instant_slopes.mean()
            
            current_raw_value = df_fault[col].iloc[-1]
            
            # ==============================================
            # محاسبه Hours_to_UCL با منطق نهایی:
            # 1. اگر شیب <= 0 -> 0
            # 2. اگر شیب > 0 و مقدار فعلی >= UCL -> 0
            # 3. اگر شیب > 0 و مقدار فعلی < UCL -> (UCL - current) / slope
            # ==============================================
            
            if avg_slope <= 0:
                hours_to_ucl = 0.0
                reason = "شیب منفی یا صفر"
            elif current_raw_value >= ucl:
                hours_to_ucl = 0.0
                reason = "مقدار فعلی از UCL بیشتر یا مساوی است"
            else:
                hours_to_ucl = (ucl - current_raw_value) / avg_slope
                reason = f"محاسبه شد: ({ucl:.4f} - {current_raw_value:.4f}) / {avg_slope:.6f}"
            
            # اطمینان از عدد بودن (NaN یا Inf نباشد)
            if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
                hours_to_ucl = 0.0
                reason = "مقدار نامعتبر (NaN/Inf)"
            else:
                hours_to_ucl = round(hours_to_ucl, 2)
            
            # چاپ دیباگ
            print(f"\n🔍 {col}:")
            print(f"   مقدار فعلی: {current_raw_value:.4f}")
            print(f"   UCL: {ucl:.4f}")
            print(f"   شیب متوسط: {avg_slope:.8f}")
            print(f"   Hours_to_UCL: {hours_to_ucl} ساعت ← {reason}")
            
            out_of_control = "No"
            if current_ewma > ucl or current_ewma < lcl:
                out_of_control = "Yes ⚠️"
            
            target_analysis_results.append({
                'Sensor': col,
                'Current_Raw_Value': round(current_raw_value, 4),
                'Current_EWMA': round(current_ewma, 4),
                'UCL': round(ucl, 4),
                'LCL': round(lcl, 4),
                'Baseline_Mean': round(mean_base, 4),
                'Baseline_Std': round(std_base, 4),
                'Average_Slope_per_Hour': round(avg_slope, 6),
                'Hours_to_UCL': hours_to_ucl,
                'Out_Of_Control': out_of_control
            })
            
        except Exception as e:
            print(f"❌ خطا در پردازش سنسور {col}: {e}")
            continue

    # ۳. رتبه‌بندی RCA
    print("\n" + "="*80)
    print("⏳ در حال محاسبه رتبه‌بندی RCA...")
    
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns:
            continue
        try:
            mean_base = df_baseline[c].mean()
            std_base = df_baseline[c].std()
            mean_fault = df_fault[c].mean()
            
            if std_base > 0:
                deviation_score = abs(mean_fault - mean_base) / std_base
            else:
                deviation_score = 0
                
            rca_list.append({
                'Sensor': c, 
                'Baseline_Mean': round(mean_base, 4),
                'Fault_Mean': round(mean_fault, 4),
                'Deviation_Score': round(deviation_score, 4)
            })
        except Exception as e:
            continue
    
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
    # ۴. ذخیره در اکسل
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)

        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            
            # اضافه کردن توضیحات روش محاسبه
            methodology_note = pd.DataFrame({
                'Parameter': [
                    'Lambda (λ)', 
                    'UCL Formula', 
                    'LCL Formula', 
                    'EWMA Formula', 
                    'Hours_to_UCL Rule'
                ],
                'Value': [
                    f'{LAMBDA}', 
                    f'μ + 3σ√(λ/(2-λ))', 
                    f'μ - 3σ√(λ/(2-λ))', 
                    'E_t = λ·X_t + (1-λ)·E_{t-1}',
                    'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0'
                ]
            })
            methodology_note.to_excel(writer, sheet_name='Methodology', index=False)

        print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
        
        # آمار نهایی
        positive_slopes = [r for r in target_analysis_results if r['Average_Slope_per_Hour'] > 0]
        positive_hours = [r for r in target_analysis_results if r['Hours_to_UCL'] > 0]
        
        print(f"\n📊 آمار نهایی Hours_to_UCL:")
        print(f"   └─ کل سنسورهای هدف: {len(target_analysis_results)}")
        print(f"   └─ سنسورهای با شیب مثبت: {len(positive_slopes)}")
        print(f"   └─ سنسورهای با زمان مثبت (در حال رسیدن): {len(positive_hours)}")
        print(f"   └─ سنسورهای با زمان صفر: {len(target_analysis_results) - len(positive_hours)}")
        
        if positive_hours:
            print(f"\n📈 سنسورهایی که در حال رسیدن به UCL هستند:")
            for res in positive_hours:
                print(f"   └─ {res['Sensor']}: {res['Hours_to_UCL']} ساعت دیگر (شیب={res['Average_Slope_per_Hour']:.6f})")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")

if __name__ == "__main__":
    run_analysis()

✅ مرحله ۱: فایل بارگذاری شد.
📅 بازه زمانی داده‌ها: 2021-03-20 17:37:26 تا 2026-05-26 12:30:11
📊 بازه سلامت: 2026-03-27 تا 2026-04-26
⚠️ بازه خطا: 2026-04-26 تا 2026-05-26
⏳ در حال محاسبه شاخص‌ها...
🔧 لاندا (λ) = 0.2

🔍 AssetID_9375:
   مقدار فعلی: 57.0000
   UCL: 51.1167
   شیب متوسط: 0.00248795
   Hours_to_UCL: 0.0 ساعت ← مقدار فعلی از UCL بیشتر یا مساوی است

🔍 AssetID_8341:
   مقدار فعلی: 0.3500
   UCL: 0.1447
   شیب متوسط: 0.00038762
   Hours_to_UCL: 0.0 ساعت ← مقدار فعلی از UCL بیشتر یا مساوی است

🔍 AssetID_8343:
   مقدار فعلی: 74.0000
   UCL: 69.4240
   شیب متوسط: 0.03274601
   Hours_to_UCL: 0.0 ساعت ← مقدار فعلی از UCL بیشتر یا مساوی است

🔍 AssetID_8344:
   مقدار فعلی: -240.0000
   UCL: -238.0078
   شیب متوسط: 0.00671824
   Hours_to_UCL: 296.54 ساعت ← محاسبه شد: (-238.0078 - -240.0000) / 0.006718

🔍 AssetID_8346:
   مقدار فعلی: 6.0000
   UCL: 6.0000
   شیب متوسط: 0.00000000
   Hours_to_UCL: 0.0 ساعت ← شیب منفی یا صفر

🔍 AssetID_9286:
   مقدار فعلی: 7.6000
   UCL: 7.6301
   شیب مت